# Explainable Job Posting Risk Analyzer — Exploratory Data Analysis (EDA)

**Author:** Academic Project Team  
**Dataset:** Employment Scam Aegean Dataset (EMSCAD) / Real or Fake Job Postings  
**Objective:** Uncover statistical distributions, class imbalance, missingness patterns, and linguistic markers in recruitment fraud data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Plotting configurations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Dataset

In [ ]:
data_path = Path("../data/raw/fake_job_postings.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(3)

## 2. Target Distribution & Class Imbalance

In [ ]:
target_counts = df["fraudulent"].value_counts()
target_pct = df["fraudulent"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=target_counts.index.map({0: "Legitimate (0)", 1: "Fraudulent (1)"}), y=target_counts.values, palette=["#2ecc71", "#e74c3c"], ax=ax[0])
ax[0].set_title("Class Distribution (Counts)")
ax[0].set_ylabel("Number of Postings")

ax[1].pie(target_counts, labels=["Legitimate (95.16%)", "Fraudulent (4.84%)"], colors=["#2ecc71", "#e74c3c"], autopct="%1.2f%%", explode=(0, 0.1))
ax[1].set_title("Class Proportions (Imbalance Ratio ~19.6 : 1)")
plt.tight_layout()
plt.show()

## 3. Missingness Across Attributes

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_cols = missing[missing > 0]

plt.figure(figsize=(10, 6))
sns.barplot(x=missing_cols.values, y=missing_cols.index, palette="Reds_r")
plt.title("Percentage of Missing Values per Feature (%)")
plt.xlabel("Missingness (%)")
plt.show()

## 4. Metadata Discriminators (Company Logo & Questions)

In [ ]:
logo_fraud = df.groupby("has_company_logo")["fraudulent"].mean() * 100
print("Fraud Rate by Company Logo Presence:")
print(f"  Without Logo: {logo_fraud[0]:.2f}% fraud rate")
print(f"  With Logo:    {logo_fraud[1]:.2f}% fraud rate")

sns.barplot(x=logo_fraud.index.map({0: "No Logo Attached", 1: "Logo Uploaded"}), y=logo_fraud.values, palette="Blues_d")
plt.title("Fraud Probability by Company Logo Indicator")
plt.ylabel("Observed Fraud Rate (%)")
plt.show()

## 5. Text Length Analysis

In [ ]:
df["desc_len"] = df["description"].fillna("").apply(len)
df["company_profile_len"] = df["company_profile"].fillna("").apply(len)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="fraudulent", y="desc_len", data=df, showfliers=False, palette=["#2ecc71", "#e74c3c"], ax=ax[0])
ax[0].set_title("Job Description Character Length by Class")
ax[0].set_xticklabels(["Legitimate", "Fraudulent"])

sns.boxplot(x="fraudulent", y="company_profile_len", data=df, showfliers=False, palette=["#2ecc71", "#e74c3c"], ax=ax[1])
ax[1].set_title("Company Profile Character Length by Class")
ax[1].set_xticklabels(["Legitimate", "Fraudulent"])
plt.tight_layout()
plt.show()

## 6. Key Takeaways for Model Engineering
1. **Severe Imbalance:** 4.84% positive class requires cost-sensitive learning (`class_weight='balanced'`) and PR-AUC metric focus.
2. **Critical Metadata Features:** Presence of company logo and company profile length are among the strongest tabular predictors.
3. **Zero-Leakage Imperative:** TF-IDF vocabulary and category encoders must only be fitted on the training partition.